# IG 계산 오차 진단 — 4장, 가중치 고정

**`lvlm_missing_ig_numerics_colab.ipynb` — L4/A100 → 기본값 → 모두 실행 → Drive 연결 허용.**
이전 코랩 재실행/추가 표시/토큰 입력은 불필요합니다. 기존 supervised 원본 ZIP 1개를 자동 검색합니다.

이전 실험은 실행 오류0이지만 IG 적분 경고가49/52개였습니다. 이번은 **기능 제작 완료, 실제 원인 진단은 아직 실행 전**입니다.
- 정답 위치나 오차 크기를 보고 고르지 않은 고정4장, 흐림/평균색 두 기준을 사용합니다.
- Midpoint32/128/512와 Gauss-Legendre128 적분, 동일 기울기의 FP32/FP64 누적을 비교합니다.
- 경로의3점에서 autograd와 중앙차분(크기3종)을 비교하고 기본 attention/SDPA MATH 계산을 기록합니다.
- FP64는 누적 정밀도만 높입니다. 네트워크는 기존 FP32 그대로이고 학습하지 않습니다.
- 실행 완료와 수치 정확성은 별개입니다. 경고/오류/원시 맵을 모두 보존합니다. 위치 정확도나 최종 Agent 성능을 평가하는 코랩은 아닙니다.

최대6,810forward/6,448backward, batch1. 중단 후 같은 설정으로 모두 실행하면 완료된 조건은 재사용합니다.
결과: **`missing_ig_numerics_results.zip`**. 경고를 없애려고 설정을 바꾸지 말고 그대로 전달하세요.

In [ ]:
import subprocess,sys
subprocess.check_call([sys.executable,'-m','pip','install','-q','torch==2.11.0',
    'transformers==5.14.0','numpy==2.1.3','Pillow==11.3.0','timm==1.0.30'])

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import base64,hashlib,json,zipfile,io
from pathlib import Path
SUP_ZIP_PATH='' # 자동 검색, 필요할 때만 원본 ZIP 경로 입력
RUN_FOLDER='missing_ig_numerics_v1'
DRIVE=Path('/content/drive/MyDrive')
SOURCE_SHA256='c7befa6f6be7c10045e74c06d5f221201f295284ff2ee9705311bce678f08974'
SOURCE_MANIFEST={'inputs/head.json': 'd9ff0610d0b74752a2dfc5b53f1cdc1db5e916e17b4260172f1c7abab8c6e7f1', 'inputs/features/57215aef477d29322472fe2877a932e2aa0635941bd29efe36a85aa5b219e872.npy': '9aa740a26e45b734213e13047b0cfe82818ecffe09cf02721d19fb680676d7b4', 'inputs/features/01871b3115e2cea822716fcb1459e7ede54894ea4442de9ee65205e82455ecc5.npy': '5cdef91629b55d0f5e7c832802cec1a92dc40e32bd5c8c92be14608e46a657dd', 'inputs/features/4e41b01147d9f1c2eb33c9e468879f8dbaedddd88d4a76ae997812326b474a07.npy': '73260de5d115e90ebe8cf66dfe134f273d7d51122cf1617faf228940f4811556', 'inputs/features/59b8fe68eb75205c8c98537a8ceda303d3fbc730858570b5c252caf253c5f20a.npy': '43d6de70ee11a9228129573054f6611e76bf668d19ed86861f7dff26134f90c3', 'inputs/features/57dd4484b64613603a2377522cf1dc198c00d04fa7b7c4e9571d48d07af4571a.npy': '8145954e807bc8383c7cbc4164a4d79e97b6217f1d8d420ab751045cede4638d', 'inputs/features/a1e808dbb67b69b94b923b107cbc6639f959189cbc2c99fb44f57c2c47039646.npy': '98d61a31858015ed73e481e2cdf5f92fc158b8a1fe87e1d6937d38f447516717', 'inputs/old_patches/dev-5b48853eb8226c7c_blur.npy': 'e8dceff28be2821a4769b3a03f8c6689d3f242be86b4d971c8da742ed59fe7f9', 'inputs/old_patches/dev-5b48853eb8226c7c_mean.npy': '84949aed00ceec9e3ccf88fa9125eb62e9287eed07848d801396f748001163cf', 'inputs/old_patches/dev-26f757507b3c93c7_blur.npy': '78ce139a44bb81b404efbadbd7d7a65b521e8bb3739e28f943de69ea67e9fd27', 'inputs/old_patches/dev-26f757507b3c93c7_mean.npy': 'efdf7c4dc2bc5b76c83a40c2a366ea37daf8872591015168de9669451adae2ef', 'inputs/old_patches/dev-3f1ef393e4238ae8_blur.npy': '7dc8d28cbc4469a443b3fa6cc31551c243ac5060df6bf72046b945c00c9b690e', 'inputs/old_patches/dev-3f1ef393e4238ae8_mean.npy': '06c3c85e47de76657842f52ee4eb368d6e3ffe1af191d92263c5a85c2b2404c0', 'inputs/old_patches/dev-54fb1c557a11f2c0_blur.npy': '1846ed12270b8f7a15b2bfd250c0c05d62aaa8980399a1bac80bfb1f54e2491d', 'inputs/old_patches/dev-54fb1c557a11f2c0_mean.npy': '32bc3133753428df53772a45dbfff83df7f1ef714a97f30f622326e8fa0f91e5', 'inputs/requests.json': '23eb5889c1985bebf324e7aeaf7add45c37d7b834c58d757ec120f6ee6cec688', 'inputs/selection.json': '1d4de8f8ce034474ad35d95803b42e21fd0eece7383603c33a3d59b0f25ace81', 'inputs/seeds.json': '47ad331414687dd3453c0a6f67cf18fdc742df781dc580df26f0cf093eb04011', 'inputs/previous32.json': '3c2e5911d4bb8d96998120d6090d8fe341c71ec917fab5b58008e9ae5a909d09', 'inputs/provenance.json': '53a48ec5634ee9ae07988958c462a91aa87a6fb357d225f6fe12ddfe6b5cd987', 'scripts/build_explainable_qa_gold.py': '291f19e6c89abf43890b74cc43643db77b0d1fc0506679e935377b44abaa5cdc', 'scripts/build_gold_v2_dev_set.py': '372907c342c47a38791f44b09710b6c662c5d56760f0f2724c36435d13ffd5bb', 'scripts/build_missing_ig_numerics_notebook.py': 'bf6019909f314da21bd47cf36bb4e7886ffc410413b3122016e8c5c5496ef2c2', 'scripts/build_retrieved_refs.py': '2b18d709a5733f07cfbaee370630d46f33fa5afe7ac80ba9005145278cb178fb', 'scripts/dev_part_crop_annotation.py': '841e16078371cf7384c428944b103c864b5be2f8827ba8ca48901aee5bf30744', 'scripts/loc_box_annotation.py': 'bc92ba581d54163019eb5b658ef91007e69164c13495ab34dec382f729340466', 'scripts/missing_ig_location.py': '0627950d8cff09f2a69194919f3b978206e80c6aa12b418c9f4ff7ae23712ce5', 'scripts/missing_ig_numerics.py': 'a9e26177b13bb6a237c77f7dc9d1d6a2f9aa1146c44169108cc926a5f30e329a', 'scripts/q3_multiregion.py': 'c431048fec6e06e6f26e2f57628e74a7861765aa4f0e63905ae16c3b40d12752', 'scripts/run_dev_prompt_diagnostic.py': 'cdca4100de8ca83d65356c2dbde4a527869a2e095957ad6329ac0f879b15cf75', 'scripts/run_gold_baselines.py': 'c5b5507c317654504725cf1c3efa9ad1e09daf00fa62d4331cb228c531a76602', 'scripts/run_gold_v2_baselines.py': '0852ea87d20951713c3b98ca67f88472f41a29481767f8947f25efd6be37f650', 'scripts/run_gold_v2_consistency_metrics.py': '802267807ee16dfa5b700f8676f2d8c294e733c99fc1586884896e164c8b3ec6', 'scripts/run_gold_v2_dev_baselines.py': '6104d142d3f52ab85e6ca130c32a4d6f1a08c7f676e199f003a2a963d9ab51de', 'scripts/run_gold_v2_q23_candidate.py': 'caad191a51a9f24685bc64e60ec407276ddfd191e572c337eb5e8909c9b6710c', 'scripts/run_gold_v2_q23_zoom_prompt_dev.py': 'c6d74f5276c22e74b43c4321acf0a7e93897499ed2346f2ae4f3f6e3302c91cb', 'scripts/run_gold_v2_q2_q3_agent_dev.py': 'd7196adf586797e9131cc52d005d6d34c30ebe3cb751ce3e8e49eade6db2d44a', 'scripts/run_gold_v2_q2_q3_refined_dev.py': 'dcd4ba3d77d2668d85098c05c889e7ea520ccf2ba4a97ac6b47dc9dbe4deafef', 'scripts/run_gold_v2_reference_ablation.py': '25b52f00cbc7911149b2377bdf0129f46cd1575831ac7f8b83dc1665534f8e6d', 'scripts/run_gold_v2_verify_agent.py': 'ded0d022e13ecf64971fd1e5d40e5595f166c1a99277837662de0e4a019126f1', 'scripts/run_missing_component_probe.py': 'b646cd82896c401dc6d756fdbd8e534d393eebacd59c19e409685ae498a2437a', 'scripts/run_missing_expert_direct_dev.py': '2251a245427edffd9cea58ced64b67185d402c8cfb2915e2926b6c33112971fd', 'scripts/run_missing_ig_location_dev.py': 'ae73f6bd349bb3149eb785fb331449d70fcd5b31c8ee7a93896668758559a22f', 'scripts/run_missing_ig_numerics.py': '9f962207cd5e0c7e4f51ef797739e8b50909aca993d8f9096f9241abba0a9f6f', 'scripts/run_q2_conflict_evidence_dev.py': '7f830216585b43d6118dfe648549cbcce8c1f78028d3675b3437431f8ab36da6', 'scripts/run_q3_missing_occlusion_dev.py': '563c85233ab7a272a5323f12732e8649a50678e7ab73903eab71e30224d9ea21', 'scripts/run_selective_missing_expert_agent_dev.py': '4cf263d5ffe7281237b99c943f2de528db9510a8a236e05017dd62d749dba5e2', 'src/insplad_benchmark/__init__.py': '8f82d2ef2a6c2a1922f1d9e1a947e24796940292bce416100e407d07a5023cad', 'src/insplad_benchmark/dataset.py': '9194614b34c3f556d9b2399b6cc896b991cde37dbb325edc5f582ee9db547874', 'src/insplad_benchmark/hf_adapters.py': '4e5e39fe437a3c7dcb52be598cbc1d1ac9acabb0b2414d8a259173a9b6d577dc', 'src/insplad_benchmark/metrics.py': '27d6b7a34515487b1ed443b671c3bc936305cb790a5b53feb750f39bfb3c3746', 'tests/test_missing_ig_numerics.py': 'a5ef8862dddc88abf158a780b7bb283784403f16e3b8779bce417b80013c1e6c'}
SOURCE_BYTES=base64.b64decode('...')
assert hashlib.sha256(SOURCE_BYTES).hexdigest()==SOURCE_SHA256
CODE=Path('/content/ig_numerics_'+SOURCE_SHA256[:12])
with zipfile.ZipFile(io.BytesIO(SOURCE_BYTES)) as archive:
    assert set(archive.namelist())==set(SOURCE_MANIFEST)
    for name in archive.namelist():
        content=archive.read(name);assert hashlib.sha256(content).hexdigest()==SOURCE_MANIFEST[name]
        path=(CODE/name).resolve();assert path.is_relative_to(CODE.resolve())
        path.parent.mkdir(parents=True,exist_ok=True)
        if path.exists():assert path.read_bytes()==content
        else:path.write_bytes(content)
(CODE/'source_manifest.json').write_text(json.dumps(SOURCE_MANIFEST,indent=2))
OUT=DRIVE/'lvlm_insplad_results'/RUN_FOLDER
if not OUT.name.startswith('missing_ig_numerics'):raise ValueError('별도 실행 폴더 이름을 유지하세요.')
RETURN_CODE=98;LOG=DRIVE/'missing_ig_numerics_console.log'
print('SOURCE',SOURCE_SHA256,'고정4장 수치진단. 가중치고정/학습0/LVLM0/gold0')

In [ ]:
def find_zip(override):
    name='supervised_fault_classification.zip';size=874753167
    if override:
        path=Path(override)
        if not path.is_file() or path.stat().st_size!=size:raise FileNotFoundError(str(path))
        return path
    for path in [DRIVE/name,DRIVE/'lvlm_insplad_data'/name]:
        if path.is_file() and path.stat().st_size==size:return path
    matches=sorted(p for p in DRIVE.rglob(name) if p.is_file() and p.stat().st_size==size)
    if not matches:raise FileNotFoundError(name)
    return matches[0]
SUP=find_zip(SUP_ZIP_PATH)
command=[sys.executable,str(CODE/'scripts/run_missing_ig_numerics.py'),
    '--capsule-dir',str(CODE),'--sup-zip',str(SUP),'--out-dir',str(OUT)]
process=subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
with LOG.open('a',encoding='utf-8') as log:
    for line in process.stdout:print(line,end='');log.write(line);log.flush()
RETURN_CODE=process.wait();print('RUNNER_RETURN_CODE',RETURN_CODE)

In [ ]:
from google.colab import files
RESULT=DRIVE/'missing_ig_numerics_results.zip'
with zipfile.ZipFile(RESULT,'w',zipfile.ZIP_DEFLATED) as archive:
    if OUT.exists():
        for path in sorted(OUT.rglob('*')):
            if path.is_file() and '_images' not in path.relative_to(OUT).parts:
                archive.write(path,'run/'+path.relative_to(OUT).as_posix())
    archive.writestr('source_bundle.zip',SOURCE_BYTES)
    archive.writestr('source_manifest.json',json.dumps(SOURCE_MANIFEST,indent=2))
    archive.writestr('runner_return_code.json',json.dumps({'return_code':RETURN_CODE}))
    if LOG.exists():archive.write(LOG,'console.log')
status=json.loads((OUT/'run_status.json').read_text()) if (OUT/'run_status.json').exists() else {'complete':False}
print('RUN STATUS',status)
if RETURN_CODE or not status.get('complete'):print('실패/미완료입니다. ZIP을 보존해 전달하세요.')
print('실행완료는수치통과가아닙니다. 경고와원시결과를함께분석합니다.')
if (OUT/'integration_summary.csv').exists():print((OUT/'integration_summary.csv').read_text())
print('ZIP',RESULT,'SHA256',hashlib.sha256(RESULT.read_bytes()).hexdigest())
files.download(str(RESULT))